# Docs 4 — Fields from Rules

Patterns that match shapes: sharp where formats are fixed, blind where
humans got creative. Today you measure both halves honestly.

In [ ]:
# The pile: six monthly reports from a community food pantry.
# Three are clean, three carry the classic damage (headers, spacing, OCR).
pile = {
 "jan.txt": """Date: 2026-01-16
Families served: 167
Donations received: $1,210.00
Contact: pantry@example.org""",
 "feb.txt": """Date: 2026-02-13
Families served: 174
Donations received: $1,385.50
Contact: pantry@example.org""",
 "mar_extracted.txt": """COMMUNITY FOOD PANTRY \u2014 MONTHLY REPORT
Page 1 of 1   PANTRY-MAR-FINAL
Date:  March 14, 2026
Families served:212
Donations received : $1,847.50
Contact: pantry@example.org""",
 "apr_extracted.txt": """Page 1 of 1   PANTRY-APR-FINAL
Date: 4/11/26
Families   served: 198
Donations received: $2,210.00
contact: pantry@example.org""",
 "may_ocr.txt": """Date: May 9, 2026
Families served: 241
Donations received: $l,655.25
Contact: pantry@example.org""",
 "jun_note.txt": """Quick note instead of the form this month, sorry! We had a
great June \u2014 somewhere around fifteen hundred dollars came in between the
two drives, and I counted 188 families across the month. \u2014 Rosa""",
}
for name, text in pile.items():
    print(f"--- {name} ({len(text)} chars)")
    print(text[:120].replace(chr(10), " / "))

In [ ]:
# Bring forward lesson 3's cleaner (condensed).
import re
def clean(text):
    text = "\n".join(l for l in text.splitlines() if not re.match(r"^Page \d+ of \d+", l))
    text = re.sub(r"  +", " ", text).replace(" :", ":")
    text = re.sub(r"([a-z]):(\d)", r"\1: \2", text)
    months = {"January":1,"February":2,"March":3,"April":4,"May":5,"June":6,
              "July":7,"August":8,"September":9,"October":10,"November":11,"December":12}
    text = re.sub(r"(" + "|".join(months) + r") (\d{1,2}), (\d{4})",
                  lambda m: f"{m.group(3)}-{months[m.group(1)]:02d}-{int(m.group(2)):02d}", text)
    text = re.sub(r"\b(\d{1,2})/(\d{1,2})/(\d{2,4})\b",
                  lambda m: f"{m.group(3) if len(m.group(3))==4 else '20'+m.group(3)}-{int(m.group(1)):02d}-{int(m.group(2)):02d}", text)
    return text

cleaned = {name: clean(raw) for name, raw in pile.items()}

## The three workhorse rules

In [ ]:
RULES = {
    "date":   re.compile(r"\b(\d{4}-\d{2}-\d{2})\b"),
    "amount": re.compile(r"\$([\d,]+\.\d\d)"),
    "email":  re.compile(r"([\w.]+@[\w.]+\.\w+)"),
    "families": re.compile(r"[Ff]amilies\s*served:\s*(\d+)"),
}

def extract(text):
    row = {}
    for field, rx in RULES.items():
        m = rx.search(text)
        row[field] = m.group(1) if m else None
    return row

table = []
for name, text in cleaned.items():
    row = extract(text)
    row["source"] = name
    table.append(row)
    print(row)

In [ ]:
# The honest scoreboard: find-rate per rule.
for field in RULES:
    found = sum(1 for r in table if r[field] is not None)
    print(f"{field:9s}: found in {found} of {len(table)} documents")

Look at the failures, not the successes. May's amount is None —
the OCR `$l,655.25` has the wrong shape (validation territory). June found
nothing at all — Rosa's note says "somewhere around fifteen hundred
dollars," and no pattern matches meaning. Quote your own three failures in
the turn-in; they are lesson 5's shopping list.

## Turn-in

The extraction table, the find-rates, and the three failure passages quoted
with one sentence each on why no pattern could match them.